# Fractional cover time series (TERN STAC API)

Load items, aggregate data over a region of iterest and plot a time series

Metadata for data used: [Seasonal Fractional Cover - Landsat, JRSRP Algorithm Version 3.0, Australia Coverage 
](https://portal.tern.org.au/metadata/TERN/0997cb3c-e2e2-45be-ac82-f5e13d24331c)

Stac Browser View: [Seasonal fractional cover - Landsat, JRSRP algorithm Version 3.0, Australia coverage
](https://stac-api.tern.org.au/stac-browser/collections/gov_qld_fractional_cover_v3_landsat_fractional_cover__seasonal?.itemFilterOpen=1)

## Before you run
- Update placeholder values (`COLLECTION_ID`, dates, bounds, point coordinates) to match your data.
- Ensure auth is configured for protected assets (for example `.netrc` and/or GDAL config).
- Install optional dependencies required by this notebook's workflow (`odc-stac`, `rioxarray`, `geopandas`, plotting extras).
- Run cells from top to bottom so variables are initialized in order.

### API Key

[Create API Key](https://ternaus.atlassian.net/wiki/spaces/TERNSup/pages/2353496065/Creating+API+Keys)

[Use API Key](https://ternaus.atlassian.net/wiki/spaces/TERNSup/pages/3355246599/Using+API+Keys+to+Access+TERN+Data+Services#Create-your-API-Key)


In [1]:
from tern_stac import TernStacClient, load_items_as_time_series, plot_time_series, load_items_odc
import geopandas as gpd

## Define some variables

These are all query parameters for STAC API to retrieve items of interest.

In [2]:
# Fill in from your catalog values
# Find the collection ID, item ID, and asset key from stac api browser
# https://stac-api.tern.org.au/stac-browser/?.language=en
COLLECTION_ID = "gov_qld_fractional_cover_v3_landsat_fractional_cover__seasonal"
START_DATE = "2020-01-01"
END_DATE = "2025-01-01"

# Bounding Box covering Brisbane area
REGION_BOUNDS = (152.914613, -27.561273, 153.142615, -27.367202)  # (minx, miny, maxx, maxy)
REGION_BOUNDS_CRS = "EPSG:4326"
# or just get the item ID and asset key from the stac api browser

## Plot the Region on an interactive map

This is to get an idea about where the bounding box we are looking exactly is.

(You may need to use jupyterlabs nbviewer to see the map rendered. https://nbviewer.org/github/ternaustralia/TERN-Data-Skills/blob/master/UQRandI2026/02_TimeSeries.ipynb)

In [10]:
# plot the bounds
import folium

# Calculate map centre for disply
minx, miny, maxx, maxy = REGION_BOUNDS
centre = [(miny + maxy) / 2, (minx + maxx) / 2]

# create interactive map
m = folium.Map(
    location=centre,
    zoom_start=11,
    # tiles="Esri.WorldImagery",
    tiles="OpenStreetMap",
)

# add region bounds to map
folium.Rectangle(
    bounds=[[miny, minx], [maxy, maxx]],
    color="red",
    weight=3,
    fill=False
).add_to(m)

# display map
m

## Find relevant Items via STAC API

We filter items by desired time range and region within specified collection

In [5]:
# instantiate Tern STAC client
client = TernStacClient()
# perforh search
search = client.search(
    collections=[COLLECTION_ID],
    datetime=f"{START_DATE}/{END_DATE}",
    bbox=[REGION_BOUNDS[0], REGION_BOUNDS[1], REGION_BOUNDS[2], REGION_BOUNDS[3]],
)
# retriev search result
items = list(search.items())
len(items)

20

For the given time range and region we have found 20 items.

## Load data using OpenDataCube Stac integration

OpendataCube Stac integration can take a list of items and open them as a single dataset.

This API also allows specifying resolution and bands of interest.

The full Australia wide data has dimensions `(band: 3, y: 135159, x: 141481)`, and many timesteps.
The filtered dataset we have covers 20 time steps and only the region of interest.

In [6]:
ds = load_items_odc(items, bands=["b1", "b2", "b3"])
ds

<xarray.Dataset> Size: 158GB
Dimensions:      (y: 54555, x: 48324, time: 20)
Coordinates:
  * y            (y) float64 436kB 7.043e+06 7.043e+06 ... 5.406e+06 5.406e+06
  * x            (x) float64 387kB -2.895e+05 -2.895e+05 ... 1.16e+06 1.16e+06
  * time         (time) datetime64[ns] 160B 2020-03-01 2020-06-01 ... 2024-12-01
    spatial_ref  int32 4B 32755
Data variables:
    b1           (time, y, x) uint8 53GB dask.array<chunksize=(1, 54555, 48324), meta=np.ndarray>
    b2           (time, y, x) uint8 53GB dask.array<chunksize=(1, 54555, 48324), meta=np.ndarray>
    b3           (time, y, x) uint8 53GB dask.array<chunksize=(1, 54555, 48324), meta=np.ndarray>

In [7]:
# display as data array (more descriptive than Dataset)
ds.to_array()

<xarray.DataArray (variable: 3, time: 20, y: 54555, x: 48324)> Size: 158GB
dask.array<stack, shape=(3, 20, 54555, 48324), dtype=uint8, chunksize=(1, 1, 54555, 48324), chunktype=numpy.ndarray>
Coordinates:
  * variable     (variable) object 24B 'b1' 'b2' 'b3'
  * time         (time) datetime64[ns] 160B 2020-03-01 2020-06-01 ... 2024-12-01
  * y            (y) float64 436kB 7.043e+06 7.043e+06 ... 5.406e+06 5.406e+06
  * x            (x) float64 387kB -2.895e+05 -2.895e+05 ... 1.16e+06 1.16e+06
    spatial_ref  int32 4B 32755

## Load time series data

This will aggregate all bands over time and produces a simple 2 dimensional dataset with 20 timesteps and one value per band per timestep.

In [8]:
ds = load_items_as_time_series(
    items,
    media_type="application/xml",  # .vrt files are XML, so use this media type to load them as xarray datasets
    role="data",
    chunks=True,
    clip_bounds=REGION_BOUNDS,
    clip_bounds_crs=REGION_BOUNDS_CRS,
    to_numpy_nodata=True,
)
ds

<xarray.DataArray (time: 20, band: 3)> Size: 480B
dask.array<concatenate, shape=(20, 3), dtype=float64, chunksize=(1, 1), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 160B 2020-03-01 2020-06-01 ... 2024-12-01
  * band         (band) int64 24B 1 2 3
    spatial_ref  int64 8B 0
Attributes:
    _FillValue:    255
    scale_factor:  1.0
    add_offset:    0.0

## Plot the extracted time series

In [ ]:
# Plot is rendered as image to display in nbviewer.
import os
import matplotlib.pyplot as plt
# ensure target directory exists
os.makedirs("images", exist_ok=True)

# plot graph to image
with plt.ioff():
    plot_time_series(
        ds.assign_coords(band=["bare soil fraction" , "non-photosynthetic vegetation fraction", "photosynthetic vegetation fraction"]),
        band_dim="band",
        figsize=(12, 6),
        compute=True,
        save_path="images/02_fractional_cover_time_series.png",
        title="Fractional cover ROI mean by band",
    )

![Fractional Cover Time Series Plot](images/02_fractional_cover_time_series.png)
